# Aggregate Loss Distribution — Motor MTPL Portfolio

This notebook combines the frequency and severity models to simulate the **aggregate loss distribution** of the portfolio using Monte Carlo simulation.

The approach follows the classical actuarial frequency-severity decomposition:

$$L_{agg} = \sum_{i=1}^{N^*} S_i$$

where:
- $N \sim \text{NegBin}(n, p)$ — number of reported claims per policyholder
- $B_j \sim \text{Bernoulli}(p_{payment})$ — whether claim $j$ results in an actual payment
- $N^* = \sum_{j=1}^{N} B_j$ — number of paid claims
- $S_i \sim \text{Lognormal/GPD}$ — severity of paid claim $i$

## 1. Imports and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import lognorm, genpareto, nbinom

# --- Frequency data ---
df_freq = pd.read_csv("freMTPL2freq.csv")
claim_counts = df_freq["ClaimNb"]
n_policyholders = claim_counts.count()
total_claims = np.sum(claim_counts)

# --- Severity data ---
df_sev = pd.read_csv("freMTPL2sev.csv")
claim_amounts = df_sev["ClaimAmount"]
n_payments = claim_amounts.count()

## 2. Frequency Model Parameters

The claim frequency follows a **Negative Binomial** distribution, chosen over Poisson due to overdispersion (variance > mean).

Parameters are estimated by moment matching.

In [ ]:
# Negative Binomial parameters — moment matching
freq_mean = claim_counts.mean()
freq_var = claim_counts.var()

p_nbinom = freq_mean / freq_var
n_nbinom = freq_mean * p_nbinom / (1 - p_nbinom)

print(f"Negative Binomial parameters:")
print(f"  n : {n_nbinom:.4f}")
print(f"  p : {p_nbinom:.4f}")

# Payment probability: proportion of claims that result in an actual payment
payment_prob = n_payments / total_claims
print(f"\nPayment probability : {payment_prob:.4f}")

## 3. Severity Model Parameters

The severity model uses a **spliced distribution**:
- **Body** (≤ €4,000) : Truncated Lognormal
- **Tail** (> €4,000) : Generalized Pareto Distribution (GPD)

The threshold of €4,000 was selected based on the Mean Excess Plot (see `severity_analysis.ipynb`).

In [ ]:
# Severity threshold
threshold = 4000

body = claim_amounts[claim_amounts <= threshold]
tail = claim_amounts[claim_amounts > threshold]
excess = tail - threshold

# Probability of being in the body vs tail
p_body = len(body) / len(claim_amounts)

# Lognormal fit on body
shape_ln, loc_ln, scale_ln = lognorm.fit(body, floc=0)

# GPD fit on tail excess
shape_gpd, loc_gpd, scale_gpd = genpareto.fit(excess, floc=0)

print(f"Body probability (below threshold): {p_body:.4f}")
print(f"Lognormal — shape: {shape_ln:.4f}, scale: {scale_ln:.4f}")
print(f"GPD — shape (xi): {shape_gpd:.4f}, scale (beta): {scale_gpd:.4f}")

## 4. Simulation Functions

The functions below are kept identical to those calibrated in `severity_analysis.ipynb`.

In [ ]:
def sample_lognormal_truncated(n, threshold, shape, scale):
    """Sample n values from a Lognormal distribution truncated at threshold."""
    samples = []
    while len(samples) < n:
        x = lognorm.rvs(shape, loc=0, scale=scale, size=n)
        x = x[x <= threshold]
        samples.extend(x.tolist())
    return np.array(samples[:n])


def sample_gpd(n, threshold, shape, scale):
    """Sample n values from a GPD shifted by threshold."""
    excess = genpareto.rvs(shape, loc=0, scale=scale, size=n)
    return threshold + excess


def simulate_severity(n):
    """Simulate n claim severities using the spliced Lognormal/GPD model."""
    u = np.random.uniform(size=n)

    n_body = np.sum(u < p_body)
    n_tail = n - n_body

    samples = np.zeros(n)
    samples[u < p_body] = sample_lognormal_truncated(
        n_body, threshold, shape_ln, scale_ln
    )
    samples[u >= p_body] = sample_gpd(
        n_tail, threshold, shape_gpd, scale_gpd
    )
    return samples

## 5. Frequency Simulation

For each Monte Carlo simulation:
1. Draw claim counts per policyholder ~ Negative Binomial
2. Apply payment probability to obtain the number of paid claims

Note: a Poisson filtered by a Binomial(p) is equivalent to applying payment_prob directly to the total — this is mathematically valid as the sum of independent Bernoulli trials.

In [ ]:
def simulate_paid_claims(n_policyholders, n_simulations):
    """
    Simulate the number of paid claims per simulation.
    
    Parameters
    ----------
    n_policyholders : int — number of policyholders in the portfolio
    n_simulations   : int — number of Monte Carlo simulations
    
    Returns
    -------
    np.array of shape (n_simulations, 1) — number of paid claims per simulation
    """
    n_paid_claims = np.zeros((n_simulations, 1), dtype=int)
    for i in range(n_simulations):
        claims_freq = np.random.negative_binomial(n_nbinom, p_nbinom, n_policyholders)
        total_freq = np.sum(claims_freq)
        n_paid_claims[i, 0] = int(payment_prob * total_freq)
    return n_paid_claims

## 6. Aggregate Loss Simulation

For each simulation, the aggregate loss is the sum of all simulated severities for the paid claims.

In [ ]:
def simulate_aggregate_loss(n_policyholders, n_simulations):
    """
    Simulate the aggregate loss distribution of the portfolio.

    Parameters
    ----------
    n_policyholders : int — number of policyholders in the portfolio
    n_simulations   : int — number of Monte Carlo simulations

    Returns
    -------
    np.array of shape (n_simulations, 1) — aggregate loss per simulation (in euros)
    """
    n_paid_claims = simulate_paid_claims(n_policyholders, n_simulations)
    aggregate_loss = np.zeros((n_simulations, 1))
    for i in range(n_simulations):
        severities = simulate_severity(int(n_paid_claims[i, 0]))
        aggregate_loss[i, 0] = np.round(np.sum(severities), 2)
    return aggregate_loss

In [ ]:
# Run simulation
# Use n_simulations=1000 for development, 10000 for final run
N_SIMULATIONS = 100

aggregate_loss = simulate_aggregate_loss(
    n_policyholders=n_policyholders,
    n_simulations=N_SIMULATIONS
)

## 7. Results

### 7.1 Descriptive Statistics

In [ ]:
aggregate_loss_M = np.round(aggregate_loss / 1_000_000, 2)

print("Aggregate Loss Distribution (in millions €)")
print(f"  Min    : {aggregate_loss_M.min():.2f}")
print(f"  Q25    : {np.percentile(aggregate_loss_M, 25):.2f}")
print(f"  Median : {np.percentile(aggregate_loss_M, 50):.2f}")
print(f"  Mean   : {aggregate_loss_M.mean():.2f}")
print(f"  Q75    : {np.percentile(aggregate_loss_M, 75):.2f}")
print(f"  Max    : {aggregate_loss_M.max():.2f}")

### 7.2 Risk Measures — Solvency II (99.5%)

- **VaR 99.5%** : the loss level not exceeded in 99.5% of simulations
- **TVaR 99.5%** : the expected loss given that we are beyond the VaR (tail expectation)

In [ ]:
var_995 = np.quantile(aggregate_loss_M, 0.995)
tvar_995 = aggregate_loss_M[aggregate_loss_M >= var_995].mean()

print("Risk Measures at 99.5% (Solvency II level)")
print(f"  VaR  99.5% : {var_995:.2f} M€")
print(f"  TVaR 99.5% : {tvar_995:.2f} M€")

### 7.3 Visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(aggregate_loss_M, bins=30, color='steelblue', edgecolor='white')
axes[0].axvline(var_995, color='red', linestyle='--', label=f'VaR 99.5% = {var_995:.2f} M€')
axes[0].axvline(tvar_995, color='orange', linestyle='--', label=f'TVaR 99.5% = {tvar_995:.2f} M€')
axes[0].set_title('Aggregate Loss Distribution')
axes[0].set_xlabel('Aggregate Loss (M€)')
axes[0].set_ylabel('Frequency')
axes[0].legend()

# Boxplot
axes[1].boxplot(aggregate_loss_M, vert=True, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.6))
axes[1].set_title('Aggregate Loss — Boxplot')
axes[1].set_ylabel('Aggregate Loss (M€)')

plt.tight_layout()
plt.show()

## 8. Model Limitations

- **Uniform λ and payment probability** : both are modelled at portfolio level. A GLM per policyholder (using characteristics such as age, vehicle type, region) would improve individual pricing accuracy.
- **Multiple payments per claim**: since each payment is treated as an independent claim, the payment probability is overstated and the severity per claim is understated. These two effects partially offset each other in the aggregate, but the severity distribution may be underestimated in the tail.
- **Lognormal fit** : The Lognormal does not fully capture the body profile — the empirical quantile curve suggests two distinct regimes, possibly driven by structural effects in the data (fixed tariffs, coverage caps, or a mix of claim types such as material vs bodily injury). Alternative distributions such as Burr or Weibull could be investigated.